# День 2 — Получение эмбеддингов (hidden states)

Цель: научиться извлекать скрытые состояния модели и считать сходство текстов.
Используем `tokenize_texts` из Дня 1.

## Задача 1: загрузка модели

In [ ]:
from transformers import AutoModel, AutoTokenizer
import torch

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()
print(model)

## Задача 2: hidden states для одного текста

In [ ]:
text = "This movie was absolutely amazing!"
tokens = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**tokens)

print(type(outputs))
print(outputs.last_hidden_state.shape)

cls_embedding = outputs.last_hidden_state[:, 0, :]
print(f'CLS embedding shape: {cls_embedding.shape}')
print(f'CLS embedding: {cls_embedding[0][:5]}...')

## Задача 3: функция получения эмбеддингов

`get_embeddings` лежит в `sentiment_lab.embeddings` и внутри вызывает
`tokenize_texts` из Дня 1: батчи по 32 текста, `no_grad`, CLS-токен,
`cpu().numpy()` и склейка через `np.vstack`.

In [ ]:
from sentiment_lab.embeddings import get_embeddings

## Задача 4: тестирование на нескольких текстах

In [ ]:
texts = [
    "This movie was absolutely amazing!",
    "Terrible movie, waste of time.",
    "Pretty good, I liked it.",
    "Boring and too long."
]

embeddings = get_embeddings(texts, tokenizer, model)
print(f'Embeddings shape: {embeddings.shape}')
print(f'Ожидается: (4, 768) для DistilBERT')

## Задача 5: сходство текстов

In [ ]:
from sentiment_lab.embeddings import similarity

sim1 = similarity("Great movie!", "Amazing film!", tokenizer, model)
sim2 = similarity("Great movie!", "Terrible film!", tokenizer, model)

print(f'Сходство похожих: {sim1:.3f}')
print(f'Сходство разных: {sim2:.3f}')